In [15]:
import pandas as pd
import duckdb
print("✅ Notebook is working with venv!")

✅ Notebook is working with venv!


# 📊 Insurance Claims — Data Exploration

This notebook explores the raw insurance dataset before building the ETL pipeline.

## Goals
- Understand the schema and data types
- Identify null values and missing data patterns
- Check for duplicates
- Spot data quality issues to address in the Silver layer
- Plan joins between the three tables

## Dataset
Insurance Claims Fraud Detection — [Kaggle source](https://www.kaggle.com/datasets/mastmustu/insurance-claims-fraud-data)

| File | Description |
|---|---|
| `insurance_data.csv` | Main claims table — 10,000 rows × 38 cols |
| `employee_data.csv` | Agents processing claims — 1,200 rows |
| `vendor_data.csv` | Vendors involved in claims — 600 rows |

In [16]:
import pandas as pd
import os

# Better display for wide dataframes
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

DATA_FOLDER = "../data"

## 1️⃣ Load All Three Datasets

In [17]:
insurance = pd.read_csv(f"{DATA_FOLDER}/insurance_data.csv")
employees = pd.read_csv(f"{DATA_FOLDER}/employee_data.csv")
vendors = pd.read_csv(f"{DATA_FOLDER}/vendor_data.csv")

print(f"Insurance: {insurance.shape[0]:,} rows × {insurance.shape[1]} cols")
print(f"Employees: {employees.shape[0]:,} rows × {employees.shape[1]} cols")
print(f"Vendors:   {vendors.shape[0]:,} rows × {vendors.shape[1]} cols")

Insurance: 10,000 rows × 38 cols
Employees: 1,200 rows × 10 cols
Vendors:   600 rows × 7 cols


## 2️⃣ Insurance Claims — Main Table

Let's take a close look at the main claims table since it's the heart of our pipeline.

In [18]:
# Quick preview
insurance.head()

,TXN_DATE_TIME,TRANSACTION_ID,CUSTOMER_ID,POLICY_NUMBER,POLICY_EFF_DT,LOSS_DT,REPORT_DT,INSURANCE_TYPE,PREMIUM_AMOUNT,CLAIM_AMOUNT,CUSTOMER_NAME,ADDRESS_LINE1,ADDRESS_LINE2,CITY,STATE,POSTAL_CODE,SSN,MARITAL_STATUS,AGE,TENURE,EMPLOYMENT_STATUS,NO_OF_FAMILY_MEMBERS,RISK_SEGMENTATION,HOUSE_TYPE,SOCIAL_CLASS,ROUTING_NUMBER,ACCT_NUMBER,CUSTOMER_EDUCATION_LEVEL,CLAIM_STATUS,INCIDENT_SEVERITY,AUTHORITY_CONTACTED,ANY_INJURY,POLICE_REPORT_AVAILABLE,INCIDENT_STATE,INCIDENT_CITY,INCIDENT_HOUR_OF_THE_DAY,AGENT_ID,VENDOR_ID
0,2020-06-01 00:00:00,TXN00000001,A00003822,PLC00008468,2015-06-23,2020-05-16,2020-05-21,Health,157.13,9000,Christopher Demarest,7701 West Saint John Road,#2010,Glendale,AZ,85308,087-11-1946,Y,54,89,Y,3,L,Own,LI,109134974,HXJP58258181908465,Bachelor,A,Major Loss,Police,0,1,GA,Savannah,4,AGENT00413,VNDR00556
1,2020-06-01 00:00:00,TXN00000002,A00008149,PLC00009594,2018-04-21,2020-05-13,2020-05-18,Property,141.71,26000,Ricardo Gatlin,8595 West 81st Drive,NaN,Arvada,CO,80005,685-33-3536,N,61,80,Y,4,L,Rent,MI,40125819,JUND46859540983731,Bachelor,A,Total Loss,Ambulance,1,0,AL,Montgomery,0,AGENT00769,VNDR00592
2,2020-06-01 00:00:00,TXN00000003,A00003172,PLC00007969,2019-10-03,2020-05-21,2020-05-26,Property,157.24,13000,Lashawn Engles,637 Britannia Drive,NaN,Vallejo,CA,94591,378-36-0672,N,47,68,Y,6,L,Rent,MI,99513168,WGZZ90128415227650,PhD,A,Total Loss,Police,0,1,CO,Grand Junction,19,AGENT00883,VNDR00031
3,2020-06-01 00:00:00,TXN00000004,A00007572,PLC00009292,2016-11-29,2020-05-14,2020-05-19,Health,172.87,16000,Steven Bassett,2803 River Drive,NaN,Thunderbolt,GA,31404,669-92-1861,Y,36,16,Y,7,L,Mortgage,MI,18429110,WIKE91555436351397,Masters,A,Minor Loss,Ambulance,0,0,GA,Savannah,12,AGENT00278,VNDR00075
4,2020-06-01 00:00:00,TXN00000005,A00008173,PLC00000204,2011-12-26,2020-05-17,2020-05-22,Travel,88.53,3000,Jason Rodriguez,7573 National Drive,NaN,Livermore,CA,94550,703-40-1033,Y,51,16,Y,2,M,Rent,HI,70752391,VYJW71311537294027,Masters,A,Major Loss,Police,0,1,TN,Nashville,18,AGENT00636,VNDR00472


In [19]:
# Column data types and non-null counts
insurance.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 38 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   TXN_DATE_TIME             10000 non-null  object 
 1   TRANSACTION_ID            10000 non-null  object 
 2   CUSTOMER_ID               10000 non-null  object 
 3   POLICY_NUMBER             10000 non-null  object 
 4   POLICY_EFF_DT             10000 non-null  object 
 5   LOSS_DT                   10000 non-null  object 
 6   REPORT_DT                 10000 non-null  object 
 7   INSURANCE_TYPE            10000 non-null  object 
 8   PREMIUM_AMOUNT            10000 non-null  float64
 9   CLAIM_AMOUNT              10000 non-null  int64  
 10  CUSTOMER_NAME             10000 non-null  object 
 11  ADDRESS_LINE1             10000 non-null  object 
 12  ADDRESS_LINE2             1495 non-null   object 
 13  CITY                      9946 non-null   object 
 14  STATE  

### 🕳 Null Analysis

Which columns have missing data and how much?

In [20]:
# Null percentages — show only columns with nulls
null_pct = (insurance.isnull().sum() / len(insurance) * 100).round(2)
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
null_pct.to_frame('null_percentage')

,null_percentage
ADDRESS_LINE2,85.05
VENDOR_ID,32.45
AUTHORITY_CONTACTED,19.45
CUSTOMER_EDUCATION_LEVEL,5.29
CITY,0.54
INCIDENT_CITY,0.46


## 3️⃣ Key Business Columns — Distributions

Understanding the distribution of key columns helps us design the Gold layer aggregations.

In [21]:
# Claim amount statistics
insurance['CLAIM_AMOUNT'].describe().round(2)

count     10000.00
mean      16563.83
std       22037.49
min         100.00
25%        2000.00
50%        7000.00
75%       21000.00
max      100000.00
Name: CLAIM_AMOUNT, dtype: float64

In [22]:
# Claim status distribution
insurance['CLAIM_STATUS'].value_counts()

CLAIM_STATUS
A    9497
D     503
Name: count, dtype: int64

In [23]:
# Insurance type distribution
insurance['INSURANCE_TYPE'].value_counts()

INSURANCE_TYPE
Property    1692
Mobile      1692
Health      1690
Life        1682
Travel      1670
Motor       1574
Name: count, dtype: int64

In [24]:
# Risk segmentation — important for our pipeline!
insurance['RISK_SEGMENTATION'].value_counts()

RISK_SEGMENTATION
L    4395
M    4150
H    1455
Name: count, dtype: int64

In [25]:
# Incident severity
insurance['INCIDENT_SEVERITY'].value_counts()

INCIDENT_SEVERITY
Total Loss    3390
Major Loss    3317
Minor Loss    3293
Name: count, dtype: int64

## 4️⃣ Employees & Vendors Tables

In [26]:
print("=== Employees Preview ===")
employees.head(2)

=== Employees Preview ===


,AGENT_ID,AGENT_NAME,DATE_OF_JOINING,ADDRESS_LINE1,ADDRESS_LINE2,CITY,STATE,POSTAL_CODE,EMP_ROUTING_NUMBER,EMP_ACCT_NUMBER
0,AGENT00001,Ray Johns,1993-06-05,1402 Maggies Way,NaN,Waterbury Center,VT,5677,34584958,HKUN51252328472585
1,AGENT00002,Angelo Borjon,2005-12-27,414 Tanya Pass,NaN,Panama City,FL,32404,107363763,OPIS19290040088204


In [27]:
print("=== Vendors Preview ===")
vendors.head(2)

=== Vendors Preview ===


,VENDOR_ID,VENDOR_NAME,ADDRESS_LINE1,ADDRESS_LINE2,CITY,STATE,POSTAL_CODE
0,VNDR00001,"King, Proctor and Jones",2027 North Shannon Drive,#5,Fayetteville,AR,72703
1,VNDR00002,Garcia Ltd,5701 East Shirley Lane,NaN,Montgomery,AL,36117


## 5️⃣ Data Quality Issues — Summary

Based on the analysis above, here's what our ETL pipeline needs to handle:

| Issue | Severity | Strategy |
|---|---|---|
| `ADDRESS_LINE2` — 85% nulls | High | Drop column entirely |
| `VENDOR_ID` — 32.5% nulls | Medium | Keep — valid that some claims have no vendor |
| `AUTHORITY_CONTACTED` — 19.4% nulls | Medium | Fill with `"None"` |
| `CUSTOMER_EDUCATION_LEVEL` — 5.3% nulls | Low | Fill with `"Unknown"` |
| `CITY` — <1% nulls across tables | Low | Drop those rows |
| Date columns stored as strings | Type issue | Convert to datetime |

## 6️⃣ Planned Joins

insurance_data
│
├── AGENT_ID   ────> employee_data (AGENT_ID)
│
└── VENDOR_ID  ────> vendor_data (VENDOR_ID)

## 7️⃣ Business Questions the Gold Layer Will Answer

1. What's the **monthly claim volume** and **total payout** trend?
2. Which **states** have the highest claim amounts?
3. Which **agents** process the most claims and what are their averages?
4. Which **vendors** appear in the highest-value claims?
5. Are there **fraud patterns** by risk segmentation or incident severity?

## ✅ Next Step

Now that we understand the data, we can build the Bronze layer to ingest these raw CSVs into DuckDB.